In [2]:
import json
import os
import sqlite3
import pandas as pd
import datetime

Initialize some varaibles that are used throughout the program

In [3]:
useful_keys = ["cveId", "dataVersion", "datePublished", "product", "vendor", "cweId", "attackComplexity", "attackVector", "availabilityImpact", "baseScore", "baseSeverity", "confidentialityImpact", "integrityImpact", "privilegesRequired", "scope", "userInteraction"]
key_tanslation = {"AV": "attackVector", "AC": "attackComplexity", "PR": "privilegesRequired", "UI": "userInteraction", "S": "scope", "C": "confidentialityImpact", "I": "integrityImpact", "A": "availabilityImpact"}
attack_vector_translation = {"N":"NETWORK", "A":"ADJACENT", "L":"LOCAL", "P":"PHYSICAL"}
other_value_translation = {"N": "NONE", "L": "LOW", "P": "PARTIAL", "C": "COMPLETE", "M": "MEDIUM", "H":"HIGH", "U": "UNCHANGED", "R": "REQUIRED"}
user_interaction_translation = {"N": "NONE", "R": "REQUIRED", "P":"PASSIVE", "A":"ACTIVE"}
empty_values = ["NULL", "n/a", "null", "n/a", 'UNKNOWN']

replace_with_none = ["availabilityImpact", "confidentialityImpact", "integrityImpact", "privilegesRequired", "userInteraction"]

years = []
for i in range(2,6):
        years.append("YEAR_202"+str(i))
columns = " ("
first = True
for i in useful_keys:
    if (i == "baseScore"):
        columns = columns + i + " REAL"
    else:
        columns = columns + i + " TEXT"
    if (first):
        columns += " PRIMARY KEY"
        first = False
    columns += ","
columns = columns[:-1] + ")"
conn = sqlite3.connect("database.sqlite")
cursor = conn.cursor()

In [3]:
def process(key, value, target):
    #print(key)
    if (isinstance(value, dict)):
        process_dict(value, target)
    elif(isinstance(value, list)):
        process_list(key, value, target)
    else:
        #print(key)
        target[key] = value


def process_dict(source, target_dict):
    for key,value in source.items():
        process(key, value, target_dict)

def process_list(key, value, target_dict):
    if (len(value) == 0):
        target_dict[key] = ""
    elif (len(value) == 1):
        process(key, value[0], target_dict)
    else:
        counter = 0
        for i in value:
            process(key+"("+str(counter)+")", i, target_dict)
            counter += 1

def process_vector_string(vector_string, target_dict):
    data = vector_string.split('/')
    for i in data:
        key = i.split(":")[0]
        value = i.split(":")[1]
        if (key in key_tanslation):
            key = key_tanslation[key]
            if (key == "attackVector"):
                value = attack_vector_translation[value]
            elif (key == "userInteraction"):
                value = user_interaction_translation[value]
            else:
                value = other_value_translation[value]
            target_dict[key] = value

Walk through every file and parse through all the data

In [36]:
for year in years:
    cursor.execute("DROP TABLE IF EXISTS "+year)
    cursor.execute("CREATE TABLE IF NOT EXISTS "+year+columns)
    
for root, dirs, files in os.walk('cves'):
    for file in files:
        file_path = os.path.join(root, file)
        year = root[5:9]
        with open(file_path) as f:
            data = json.load(f)
            data_dict = {}
            process_dict(data, data_dict)
            insert_dict = {}
            for key in useful_keys:
                if (key in data_dict):
                    if (data_dict[key] in empty_values):
                        if (key in replace_with_none):
                            insert_dict[key] = "NONE"
                        elif(key == "baseSeverity" and (not data_dict["baseScore"] == None) and (not data_dict["baseScore"] in empty_values)):
                            base_score = int(data_dict["baseScore"])
                            if base_score == 0:
                                insert_dict["baseSeverity"]="NONE"
                            elif ((base_score > 0) and (base_score < 4)) :
                                insert_dict["baseSeverity"]="LOW"
                            elif ((base_score > 3.9) and (base_score < 7)):
                                insert_dict["baseSeverity"]="MEDIUM"
                            elif ((base_score > 6.9) and (base_score < 9)):
                                insert_dict["baseSeverity"]="HIGH"
                            elif base_score <= 10:
                                insert_dict["baseSeverity"]="CRITICAL"
                            else:
                                print(base_score)
                        else:
                            insert_dict[key] = None
                    else:
                        insert_dict[key] = data_dict[key]
                    
                else:
                    insert_dict[key] = None
            if (None in insert_dict.values() and ("vectorString" in data_dict.keys()) and (data_dict["vectorString"]!="null")):
                process_vector_string(data_dict["vectorString"], insert_dict)
            insert_query = "INSERT INTO YEAR_"+str(year)+" ("+(', '.join(useful_keys))+") "+"VALUES ("+("?, "*len(useful_keys))[:-2]+")"
            #print(insert_query)
            cursor.execute(insert_query, list(insert_dict.values()))
    conn.commit()

If datePublished is NULL, usually means that the cve was not accepted and for the rest of the columns of the row are also NULL

In [37]:
for year in years:
        cursor.execute("DELETE FROM "+year+" WHERE datePublished IS NULL")
        conn.commit()

In [40]:
year_counts = {}
for year in years:
    year_counts[year] = {}
    for i in range(1, 13):
        year_counts[year][i] = 0
    cursor.execute("SELECT datePublished FROM "+year)
    data = cursor.fetchall()
    for row in data:
        date_string = row[0][:19]
        date = datetime.datetime.strptime(date_string, "%Y-%m-%dT%H:%M:%S")
        year_counts[year][date.month]+=1
    year_counts[year]["total"]=len(data)
    #print(len(data))
#print(year_counts)
   

26721
30859
38781
43707
{'YEAR_2022': {1: 2094, 2: 2582, 3: 1940, 4: 1898, 5: 2448, 6: 2228, 7: 2092, 8: 2367, 9: 2362, 10: 2124, 11: 2077, 12: 2509, 'total': 26721}, 'YEAR_2023': {1: 2304, 2: 2135, 3: 2485, 4: 2313, 5: 3375, 6: 2380, 7: 2191, 8: 2321, 9: 2455, 10: 2938, 11: 2441, 12: 3521, 'total': 30859}, 'YEAR_2024': {1: 3339, 2: 2881, 3: 3613, 4: 3516, 5: 3823, 6: 2877, 7: 3048, 8: 2813, 9: 2505, 10: 3410, 11: 3822, 12: 3134, 'total': 38781}, 'YEAR_2025': {1: 3911, 2: 2908, 3: 3590, 4: 3978, 5: 3585, 6: 3614, 7: 3760, 8: 3515, 9: 3826, 10: 3705, 11: 2928, 12: 4387, 'total': 43707}}


In [41]:
for year in years:
        cursor.execute("DELETE FROM "+year+" WHERE baseScore IS NULL")
        conn.commit()

TODO, get counts for each year based on cweID, attackComplexity, attackVector, availabilityImpact, baseSeverity, confidentialityImpact, integrityImpact, privilegesRequired, scope, and userInteraction.

Likely stacked bar chart would work well, look at previous code to see how can be done